# FactIndic prompt-pilot NLI screening (Colab)

This notebook scores the **152-row quality-filtered prompt pilot** with `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`. It does not regenerate summaries. Upload `factindic_pilot_quality_filtered_candidate.zip` from the project’s `results/prompt_pilot_review_candidate_2026-11-09/` folder.

The model compares each summary with chunks of the exact `source_visible` text. Scores are **automated screening signals only**. They are not factuality labels, human verification, or proof that a summary is correct. Do not present them as gold annotations. The ZIP includes the unmodified human review sheet with blank labels.

Use a Colab **GPU runtime**. Run cells from top to bottom.


In [ ]:
# Install a compatible, pinned Transformers 4.x stack. No datasets package is needed.
!pip -q install "transformers==4.57.3" "huggingface-hub==0.36.2" "sentencepiece>=0.2,<1" "safetensors>=0.4" "accelerate>=1.2,<2"


In [ ]:
from google.colab import files
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, json, math, os, shutil, zipfile, collections
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification

assert torch.cuda.is_available(), "In Colab: Runtime > Change runtime type > T4 GPU, then reconnect and run from the top."
print("GPU:", torch.cuda.get_device_name(0))
print("Transformers:", transformers.__version__)

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError(f"Upload exactly one candidate ZIP file; received: {list(uploaded)}")
INPUT_ZIP = Path('/content') / zip_names[0]
WORK = Path('/content/factindic_nli_screening')
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(parents=True)
with zipfile.ZipFile(INPUT_ZIP) as zf:
    generation_entries = [n for n in zf.namelist() if Path(n).name == 'generations.csv']
    review_entries = [n for n in zf.namelist() if Path(n).name == 'human_review.csv']
    if len(generation_entries) != 1:
        raise ValueError(f"Expected one generations.csv in ZIP; found {generation_entries}")
    zf.extract(generation_entries[0], WORK)
    if len(review_entries) == 1:
        zf.extract(review_entries[0], WORK)
GEN_PATH = WORK / generation_entries[0]
with GEN_PATH.open(encoding='utf-8-sig', newline='') as f:
    reader = csv.DictReader(f)
    original_fields = reader.fieldnames or []
    rows = list(reader)
required = {'condition','id','arm','source_visible','prediction'}
missing = required - set(original_fields)
if missing:
    raise ValueError(f"Candidate generations.csv is missing columns: {sorted(missing)}")
if len(rows) != 152:
    raise ValueError(f"Expected the reviewed 152-row quality-filtered candidate; found {len(rows)} rows. Check the uploaded ZIP.")
if any(not row['source_visible'].strip() or not row['prediction'].strip() for row in rows):
    raise ValueError("Found an empty source_visible or prediction; stopping without scoring.")
print("Loaded", len(rows), "rows")
print("Conditions:", dict(collections.Counter(r['condition'] for r in rows)))
print("Prompt arms:", dict(collections.Counter(r['arm'] for r in rows)))


In [ ]:
MODEL_ID = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
MAX_LENGTH = 512
SOURCE_CHUNK_TOKENS = 360
CHUNK_OVERLAP_TOKENS = 48
BATCH_SIZE = 12

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, low_cpu_mem_usage=True
).to("cuda")
model.eval()

label_map = {int(k): str(v).lower() for k, v in model.config.id2label.items()}
def class_index(name):
    matches = [i for i, label in label_map.items() if name in label]
    if len(matches) != 1:
        raise ValueError(f"Could not uniquely map NLI label {name!r}; model labels are {label_map}")
    return matches[0]
ENTAILMENT_ID = class_index("entailment")
CONTRADICTION_ID = class_index("contradiction")
NEUTRAL_ID = class_index("neutral")
print("Device: cuda | Model labels:", label_map)


In [ ]:
def token_chunks(text, chunk_size=SOURCE_CHUNK_TOKENS, overlap=CHUNK_OVERLAP_TOKENS):
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    if not token_ids:
        return [""]
    step = chunk_size - overlap
    chunks = []
    for start in range(0, len(token_ids), step):
        piece = token_ids[start:start + chunk_size]
        if not piece:
            break
        chunks.append(tokenizer.decode(piece, skip_special_tokens=True))
        if start + chunk_size >= len(token_ids):
            break
    return chunks

all_pairs = []
row_chunk_ranges = []
for row_index, row in enumerate(rows):
    chunks = token_chunks(row['source_visible'])
    hypo_tokens = tokenizer.encode(row['prediction'], add_special_tokens=False)
    # Preserve the complete summary as the hypothesis; fail explicitly if it cannot fit.
    if len(hypo_tokens) > MAX_LENGTH - 8:
        raise ValueError(f"Summary at row {row_index} is too long for the model hypothesis budget ({len(hypo_tokens)} tokens).")
    begin = len(all_pairs)
    for chunk_index, chunk in enumerate(chunks):
        all_pairs.append((row_index, chunk_index, chunk, row['prediction']))
    row_chunk_ranges.append((begin, len(all_pairs)))
print(f"Prepared {len(all_pairs)} source-summary pairs for {len(rows)} summaries.")

scored_pairs = []
with torch.inference_mode():
    for start in range(0, len(all_pairs), BATCH_SIZE):
        batch = all_pairs[start:start + BATCH_SIZE]
        premises = [item[2] for item in batch]
        hypotheses = [item[3] for item in batch]
        encoded = tokenizer(
            premises, hypotheses, return_tensors="pt", padding=True,
            truncation="only_first", max_length=MAX_LENGTH
        )
        encoded = {key: value.to("cuda") for key, value in encoded.items()}
        probs = torch.softmax(model(**encoded).logits.float(), dim=-1).cpu().tolist()
        for item, probability in zip(batch, probs):
            scored_pairs.append({
                'row_index': item[0], 'chunk_index': item[1], 'chunk_text': item[2],
                'entailment': float(probability[ENTAILMENT_ID]),
                'contradiction': float(probability[CONTRADICTION_ID]),
                'neutral': float(probability[NEUTRAL_ID]),
                'sequence_tokens': int(encoded['input_ids'].shape[1]),
            })
        done = min(start + len(batch), len(all_pairs))
        if done == len(all_pairs) or done % (BATCH_SIZE * 10) == 0:
            print(f"Scored {done}/{len(all_pairs)} source-summary pairs")

for i, (begin, end) in enumerate(row_chunk_ranges):
    group = scored_pairs[begin:end]
    best = max(group, key=lambda item: item['entailment'])
    rows[i].update({
        'nli_model': MODEL_ID,
        'nli_max_entailment': round(best['entailment'], 6),
        'nli_contradiction_at_best_entailment': round(best['contradiction'], 6),
        'nli_neutral_at_best_entailment': round(best['neutral'], 6),
        'nli_max_contradiction_any_chunk': round(max(x['contradiction'] for x in group), 6),
        'nli_best_source_chunk_index': best['chunk_index'],
        'nli_best_source_chunk': best['chunk_text'],
        'nli_source_chunks': len(group),
        'nli_hypothesis_tokens': len(tokenizer.encode(rows[i]['prediction'], add_special_tokens=False)),
        'nli_hypothesis_truncated': False,
        'nli_max_pair_sequence_tokens': max(x['sequence_tokens'] for x in group),
    })
print("Finished NLI screening for", len(rows), "summaries.")


In [ ]:
# Export the per-summary scores, a descriptive aggregate, provenance, and the original blank review sheet.
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT_DIR = Path('/content') / f'factindic_nli_screening_{STAMP}'
OUT_DIR.mkdir(parents=True, exist_ok=True)
score_path = OUT_DIR / 'nli_screened_generations.csv'
score_fields = list(rows[0].keys())
with score_path.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=score_fields)
    writer.writeheader(); writer.writerows(rows)

summary_rows = []
for condition in sorted(set(r['condition'] for r in rows)):
    for arm in sorted(set(r['arm'] for r in rows)):
        group = [r for r in rows if r['condition'] == condition and r['arm'] == arm]
        if not group: continue
        summary_rows.append({
            'condition': condition, 'arm': arm, 'n': len(group),
            'mean_max_entailment': round(sum(float(r['nli_max_entailment']) for r in group)/len(group), 6),
            'median_max_entailment': round(sorted(float(r['nli_max_entailment']) for r in group)[len(group)//2], 6),
            'mean_contradiction_at_best_entailment': round(sum(float(r['nli_contradiction_at_best_entailment']) for r in group)/len(group), 6),
            'mean_max_contradiction_any_chunk': round(sum(float(r['nli_max_contradiction_any_chunk']) for r in group)/len(group), 6),
        })
summary_path = OUT_DIR / 'nli_screening_descriptive_summary.csv'
with summary_path.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
    writer.writeheader(); writer.writerows(summary_rows)

# Matched within-document prompt comparison. This makes no factuality claim.
pair_map = {}
for row in rows:
    key = (row['condition'], row['id'])
    pair_map.setdefault(key, {})[row['arm']] = row
paired_rows = []
for (condition, example_id), arms in sorted(pair_map.items()):
    if set(arms) != {'standard', 'factuality_aware'}:
        raise ValueError(f"Unmatched or duplicate prompt arms for {condition}/{example_id}: {sorted(arms)}")
    standard = arms['standard']; aware = arms['factuality_aware']
    paired_rows.append({
        'condition': condition, 'id': example_id, 'subset': standard.get('subset', ''),
        'standard_max_entailment': float(standard['nli_max_entailment']),
        'factuality_aware_max_entailment': float(aware['nli_max_entailment']),
        'aware_minus_standard_max_entailment': float(aware['nli_max_entailment']) - float(standard['nli_max_entailment']),
        'standard_contradiction_at_best_entailment': float(standard['nli_contradiction_at_best_entailment']),
        'factuality_aware_contradiction_at_best_entailment': float(aware['nli_contradiction_at_best_entailment']),
        'aware_minus_standard_contradiction_at_best_entailment': float(aware['nli_contradiction_at_best_entailment']) - float(standard['nli_contradiction_at_best_entailment']),
    })
paired_path = OUT_DIR / 'nli_screening_paired_prompt_comparison.csv'
with paired_path.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(paired_rows[0].keys()))
    writer.writeheader(); writer.writerows(paired_rows)
paired_summary = []
for condition in sorted(set(r['condition'] for r in paired_rows)):
    group = [r for r in paired_rows if r['condition'] == condition]
    entail_deltas = [r['aware_minus_standard_max_entailment'] for r in group]
    contradiction_deltas = [r['aware_minus_standard_contradiction_at_best_entailment'] for r in group]
    paired_summary.append({
        'condition': condition, 'n_matched_documents': len(group),
        'mean_aware_minus_standard_max_entailment': round(sum(entail_deltas)/len(group), 6),
        'median_aware_minus_standard_max_entailment': round(__import__('statistics').median(entail_deltas), 6),
        'aware_entailment_score_higher': sum(d > 0 for d in entail_deltas),
        'aware_entailment_score_lower': sum(d < 0 for d in entail_deltas),
        'equal_entailment_scores': sum(d == 0 for d in entail_deltas),
        'mean_aware_minus_standard_contradiction_at_best_entailment': round(sum(contradiction_deltas)/len(group), 6),
    })
paired_summary_path = OUT_DIR / 'nli_screening_paired_prompt_summary.csv'
with paired_summary_path.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(paired_summary[0].keys()))
    writer.writeheader(); writer.writerows(paired_summary)
print('Matched prompt pairs:', len(paired_rows))
print('By-condition paired summary:', paired_summary)

input_sha256 = hashlib.sha256(INPUT_ZIP.read_bytes()).hexdigest()
metadata = {
    'status': 'automated_nli_screening_only_not_factuality_gold_labels',
    'input_zip': INPUT_ZIP.name, 'input_zip_sha256': input_sha256,
    'n_rows': len(rows), 'model': MODEL_ID, 'model_labels': label_map,
    'transformers_version': transformers.__version__, 'torch_version': torch.__version__,
    'device': torch.cuda.get_device_name(0), 'max_length': MAX_LENGTH,
    'source_chunk_tokens': SOURCE_CHUNK_TOKENS, 'chunk_overlap_tokens': CHUNK_OVERLAP_TOKENS,
    'batch_size': BATCH_SIZE, 'hypothesis_truncated': False,
    'scores_are_human_factuality_labels': False,
    'interpretation': 'Entailment/contradiction probabilities are model signals only; not calibrated factuality judgments.',
    'created_utc': STAMP,
}
(OUT_DIR / 'nli_screening_metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
# Carry forward the candidate's review worksheet unchanged; do not fill human-label fields.
with zipfile.ZipFile(INPUT_ZIP) as zf:
    for name in zf.namelist():
        if Path(name).name == 'human_review.csv':
            (OUT_DIR / 'human_review.csv').write_bytes(zf.read(name))
            break
zip_path = Path('/content') / f'factindic_nli_screening_{STAMP}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUT_DIR.iterdir()): zf.write(path, arcname=f'{OUT_DIR.name}/{path.name}')
print('Saved screening package:', zip_path)
print('Rows:', len(rows), '| summary groups:', len(summary_rows))
files.download(str(zip_path))


## How to interpret the output

- `nli_screened_generations.csv` contains each summary, its condition/prompt arm, the exact visible source text, and the NLI probabilities.
- `nli_screening_descriptive_summary.csv` reports simple per-condition/per-arm averages and medians. `nli_screening_paired_prompt_comparison.csv` and `nli_screening_paired_prompt_summary.csv` compare the two prompt arms within the same document. Treat all of these as exploratory summaries of model scores; do not infer statistical significance or factuality from them.
- `nli_best_source_chunk` shows which source chunk gave the highest entailment score. A high score does not prove all claims are supported; a low score can reflect wording or language coverage.
- `human_review.csv` is carried forward unchanged. The notebook does not create human labels.
